<a href="https://colab.research.google.com/github/rakesh-mandal/ML/blob/main/cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

# 1. Load and split data
df = pd.read_csv('train.csv')
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define preprocessing pipelines for different column types
numerical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# 3. Combine them into a single ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, ['Age']),
    ('cat', categorical_pipeline, ['Sex', 'Embarked'])
], remainder='passthrough') # Pclass, SibSp, Parch, Fare pass through un-imputed but scale them if needed

# 4. Create the final master pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=chi2, k=8)),
    ('classifier', DecisionTreeClassifier())
])

# Train and predict safely!
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [2]:
from sklearn.metrics import accuracy_score

In [3]:
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

Accuracy: 0.7932960893854749
